# 第十一章：RAG 系统实战

## 检索增强生成——让 LLM 说真话的核心范式

LLM 的知识截止于训练数据的时间点，且容易产生**幻觉**（hallucination）。**检索增强生成 (Retrieval-Augmented Generation, RAG)** 通过从外部知识库检索相关文档来增强 LLM 的回答质量：

```
用户问题 → 检索相关文档 → 文档+问题 → LLM → 基于证据的回答
```

本章从零构建完整的 RAG 系统。

## 11.1 Embedding 与语义搜索

**Embedding** 将文本映射到高维向量空间。语义相似的文本在向量空间中距离更近：

$$\text{sim}(\mathbf{v}_1, \mathbf{v}_2) = \frac{\mathbf{v}_1 \cdot \mathbf{v}_2}{|\mathbf{v}_1| |\mathbf{v}_2|}$$

现代 Embedding 模型通过**对比学习**训练：正样本对（语义等价的文本）拉近，负样本对推远，使用 InfoNCE Loss 优化。

| 模型 | 维度 | 特点 |
|------|------|------|
| `text-embedding-3-small` | 512/1536 | OpenAI，多语言，性价比高 |
| `bge-large-zh-v1.5` | 1024 | BAAI，中文检索 SOTA |
| `all-MiniLM-L6-v2` | 384 | 轻量，本地部署 |

In [ ]:
# === Embedding 实战：语义搜索 ===
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    "反向传播是神经网络中通过链式法则计算梯度的算法",
    "CNN 通过卷积核提取局部特征，池化层降低空间维度",
    "Transformer 使用多头自注意力机制并行处理序列",
    "梯度下降沿损失函数的负梯度方向迭代更新参数",
]
doc_emb = model.encode(documents, normalize_embeddings=True)

query = "如何使用链式法则训练神经网络？"
q_emb = model.encode(query, normalize_embeddings=True)

# 余弦相似度检索
scores = np.dot(q_emb, doc_emb.T)
for i in np.argsort(scores)[::-1][:3]:
    print(f"  score={scores[i]:.3f} | {documents[i]}")


## 11.2 向量数据库

百万级以上文档库无法暴力搜索 (O(n·d))。向量数据库通过**近似最近邻 (ANN)** 索引实现亚线性检索。

### 主流索引算法

| 算法 | 原理 | 特点 |
|------|------|------|
| **HNSW** | 分层可导航小世界图 | 高召回，内存消耗大 |
| **IVF** | 倒排文件索引，先聚类再搜索 | 速度快，`nprobe` 调节召回 |
| **PQ** | 乘积量化，压缩向量 | 极致压缩，精度有损 |

### 常用向量数据库

| 数据库 | 定位 |
|--------|------|
| **Chroma** | 轻量嵌入式，原型首选 |
| **FAISS** | Meta 开源，十亿级高性能检索 |
| **Milvus** | 分布式生产级，水平扩展 |
| **Qdrant** | Rust 实现，过滤+向量混合查询 |

### 混合检索

纯向量检索在专有名词、精确 ID 等场景不如关键词匹配。**BM25** (基于 TF-IDF 改进) + 稠密向量检索 = 互补短板。

In [ ]:
# === Chroma 向量数据库最小示例 ===
# pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="ai_knowledge", embedding_function=ef
)

collection.add(
    documents=[
        "反向传播通过链式法则计算梯度。",
        "CNN 使用卷积核提取图像局部特征。",
        "Transformer 基于自注意力并行处理序列。",
    ],
    ids=["d1", "d2", "d3"],
)

results = collection.query(query_texts=["神经网络如何学习？"], n_results=2)
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"  dist={dist:.3f} | {doc}")


## 11.3 RAG 管道

### 标准三阶段

```
检索 (Retrieve): embedding 相似度 → top-k 文档
    ↓
增强 (Augment): 将文档拼入 prompt 模板，标注来源
    ↓
生成 (Generate): LLM 基于证据回答
```

### Chunking——RAG 的质量瓶颈

| 策略 | 做法 | 推荐场景 |
|------|------|---------|
| **固定大小** | 每 N 字符一刀切 | 快速原型 |
| **递归分割** | 按 `\n\n`→`\n`→`。` 优先级切 | 通用 |
| **语义分割** | embedding 相似度变化检测主题边界 | 高质量需求 |
| **句子窗口** | 检索时返回前后窗口 | 需要上下文 |

**500~1000 字符**是大多数场景的甜蜜点。

### 检索→重排序→生成

粗筛 (bi-encoder embedding top-50) → 精排 (Cross-Encoder 逐对打分 top-5) → 生成。

Cross-Encoder 同时接收 query 和 document，输出相关性分数——比 embedding 更准确但慢得多，仅用于精排。

In [ ]:
# === 简易 RAG 管道 ===
class SimpleRAG:
    def __init__(self, documents, chunk_size=500):
        from sentence_transformers import SentenceTransformer
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.chunks = self._chunk(documents, chunk_size)
        self.embeddings = self.encoder.encode(self.chunks, normalize_embeddings=True)
    
    def _chunk(self, docs, size):
        chunks = []
        for doc in docs:
            sents = doc.replace('！','。').replace('？','。').split('。')
            cur = ""
            for s in sents:
                if len(cur) + len(s) > size and cur:
                    chunks.append(cur.strip())
                    cur = s
                else:
                    cur += s + "。"
            if cur.strip(): chunks.append(cur.strip())
        return chunks
    
    def retrieve(self, query, top_k=3):
        import numpy as np
        q_emb = self.encoder.encode(query, normalize_embeddings=True)
        scores = np.dot(q_emb, self.embeddings.T)
        top = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], scores[i]) for i in top]
    
    def build_prompt(self, query, retrieved):
        ctx = "\n\n---\n\n".join(f"[{i+1}] {c}" for i,(c,_) in enumerate(retrieved))
        return f"基于以下文档回答。如无足够信息请说明。\n\n{ctx}\n\n问题: {query}\n回答:"

# 测试
rag = SimpleRAG([
    "反向传播通过链式法则计算神经网络中每个参数的梯度。梯度下降利用这些梯度更新参数。",
    "CNN 通过卷积层提取局部特征，池化层降低维度。参数共享使 CNN 参数量远小于 MLP。",
    "Transformer 使用自注意力机制替代 RNN。多头注意力允许模型关注不同表示子空间。",
])
print(rag.build_prompt("神经网络如何学习？", rag.retrieve("神经网络如何学习？")))


## 11.4 高级 RAG 变体

### Self-RAG

引入**自我反思**：判断是否需要检索 → 逐段评估相关性 → 生成后验证是否有依据。

Google DeepMind 通过特殊 reflection token 控制流程。

### Corrective RAG (CRAG)

检索后增加评估-纠正：文档不相关 → 改写查询 → 重新检索 → 生成。适用于知识库覆盖不全的场景。

### Graph RAG

预构建知识图谱（实体-关系），检索时沿图遍历获取子图上下文。在多跳推理和全局摘要任务中显著优于传统 RAG。Microsoft GraphRAG 先将文档聚类 → LLM 生成社区摘要 → 混合检索。

### RAG 评估 (RAGAS)

| 指标 | 含义 |
|------|------|
| **Faithfulness** | 回答是否忠实于检索文档 |
| **Answer Relevance** | 回答是否直接回应问题 |
| **Context Recall** | 检索是否覆盖了答案所需信息 |
| **Context Precision** | 检索结果中相关文档比例 |

## 本章小结

| 层次 | 技术 | 关键点 |
|------|------|--------|
| **Embedding** | 语义向量化 | 对比学习，余弦相似度 |
| **向量数据库** | Chroma/FAISS | ANN 索引，HNSW/IVF |
| **RAG 管道** | 检索→增强→生成 | Chunking 500-1000 字符 |
| **高级 RAG** | Self-RAG/Graph RAG | 反思机制、知识图谱增强 |

> **下一章**：第十二章——AI Agent，让 LLM 不仅能回答问题，还能自主执行任务。